In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, to_date
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

In [2]:
spark = (
    SparkSession.builder
    .appName("yahoo_finance")
    .master("spark://spark-master:7077")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

In [3]:
# -----------------------------
# Kafka source (batch read)
# -----------------------------
kafka_df = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "yahoo_finance") \
    .option("startingOffsets", "earliest") \
    .load()

In [4]:
# Kafka value is binary -> string
raw_df = kafka_df.selectExpr("CAST(value AS STRING) as json_str")

In [5]:
# -----------------------------
# Schema Yahoo Finance
# -----------------------------
schema = StructType([
    StructField("symbol", StringType(), True),
    StructField("date", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("return", DoubleType(), True),
    StructField("volatility", DoubleType(), True)
])

parsed_df = raw_df.select(
    from_json(col("json_str"), schema).alias("data")
).select("data.*")

In [6]:
# Cast date properly
parsed_df = parsed_df.withColumn("date", to_date(col("date")))

In [7]:
# -----------------------------
# Simple validation / transform
# -----------------------------
batch_df = parsed_df.filter(col("price").isNotNull())

batch_df.printSchema()
batch_df.show(10, truncate=False)

root
 |-- symbol: string (nullable = true)
 |-- date: date (nullable = true)
 |-- price: double (nullable = true)
 |-- return: double (nullable = true)
 |-- volatility: double (nullable = true)

+------+----------+------------------+---------------------+--------------------+
|symbol|date      |price             |return               |volatility          |
+------+----------+------------------+---------------------+--------------------+
|AAPL  |2023-02-01|143.13467407226562|0.007900769124810303 |0.012757897978624111|
|AAPL  |2023-02-02|148.43963623046875|0.03706273265082349  |0.014354441708978927|
|AAPL  |2023-02-03|152.06150817871094|0.024399628294823117 |0.013969191930132598|
|AAPL  |2023-02-06|149.33523559570312|-0.017928748804751704|0.013955409530839749|
|AAPL  |2023-02-07|152.20913696289062|0.01924463008159738  |0.014142214989727894|
|AAPL  |2023-02-08|149.5222625732422 |-0.017652517077890728|0.015311689065079424|
|AAPL  |2023-02-09|148.48878479003906|-0.00691186560059498 |0.01529

In [8]:
hadoop_conf = spark._jsc.hadoopConfiguration()

hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.access.key", "minio")
hadoop_conf.set("fs.s3a.secret.key", "minio123")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")

In [10]:
# -----------------------------
# Sink (example: Parquet)
# -----------------------------
batch_df.write \
    .mode("overwrite") \
    .parquet("s3a://data-lake/curated/yahoo_finance")

print("✅ Spark batch job finished")

✅ Spark batch job finished
